# 03-3. Lead baseline

이 노트북에서 하는 일: 뉴스 원문의 앞 50어절 또는 80어절을 그대로 요약으로 사용하고, 원본 test summary와 비교한다.

이 실험은 모델을 사용하지 않는 규칙 기반 기준선입니다.

- 학습과 파인튜닝을 하지 않는다.
- 원본 test 2,740건을 그대로 사용한다.
- `summary`는 Lead 요약 생성에 사용하지 않고 ROUGE 평가에만 사용한다.
- 이 데이터는 문장부호가 부족하여 Lead-3 문장 대신 공백 기준 50·80어절을 비교한다.
- 이후 KoBART zero-shot 및 파인튜닝 결과가 단순한 기사 앞부분보다 좋은지 판단하는 기준이 된다.


## 1. 환경 및 설정

In [2]:
import os
import sys
from pathlib import Path

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert PROJECT.exists(), "프로젝트 경로를 확인하세요."
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), (
    "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
)

os.environ["HF_HOME"] = str(PROJECT / ".cache" / "huggingface")
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

RESULTS = PROJECT / "results" / "03-3_baseline_lead"
RESULTS.mkdir(parents=True, exist_ok=True)

SEED = 42
DATASET_ID = "daekeun-ml/naver-news-summarization-ko"
LEAD_LENGTHS = [50, 80]

print("Python:", sys.version.split()[0])
print("결과 폴더:", RESULTS)

Python: 3.11.9
결과 폴더: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\results\03-3_baseline_lead


In [3]:
import json
import random

import evaluate
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi

random.seed(SEED)
np.random.seed(SEED)

## 2. 원본 test 데이터 로드

In [4]:
api = HfApi()
dataset_revision = api.dataset_info(DATASET_ID).sha
raw_dataset = load_dataset(DATASET_ID, revision=dataset_revision)

test_df = raw_dataset["test"].to_pandas().reset_index(drop=True)
test_df.insert(0, "source_index", np.arange(len(test_df), dtype=np.int64))
test_df.insert(0, "row_id", "test:" + test_df["source_index"].astype(str))

assert len(test_df) == 2740
assert test_df[["document", "summary"]].notna().all().all()
assert test_df["document"].str.strip().ne("").all()
assert test_df["summary"].str.strip().ne("").all()

print("dataset revision:", dataset_revision)
print("test rows:", len(test_df))
display(test_df[["title", "document", "summary"]].head(3))

dataset revision: 916dfe6aed651b2ecd9d04fa0a704f8953e90762
test rows: 2740


,title,document,summary
0,아이트로닉스 차량용 복합기능형 졸음 방지 단말기 특허 출원,아이엘사이언스의 자회사 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출...,"아이일, 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출원했다고 4일 ..."
1,뉴스페이스로 가는 한국… 우주기업 성장할 기술·터전 만든다,한국형 우주발사체 누리호 가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 날...,한국형 우주발사체 누리호가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 성공...
2,SP500 올 상반기 21%↓…50년래 최악 월가월부,닥터둠 루비니 주가 50% 폭락할 것 뉴욕 증시 3대지수 모두 하락 대형주 중심의 ...,대 대형주 중심의 미국 스탠더드앤드푸어스 S P 500 지수가 올 상반기 50여년만...


## 3. Lead-50·Lead-80 생성

가이드에서는 일반적인 Lead-3 대신 단어 수 기준을 권장합니다. 한국어에서는 공백으로 나눈 단위를 정확히는 **어절**이라고 하므로, 이 노트북에서는 앞 50어절과 80어절을 사용합니다.


In [5]:
def make_lead_summary(document, word_limit):
    words = str(document).split()
    return " ".join(words[:word_limit])

result_df = test_df[[
    "row_id", "title", "document", "summary", "category", "press", "link"
]].copy()

for word_limit in LEAD_LENGTHS:
    result_df[f"lead_{word_limit}"] = result_df["document"].map(
        lambda document: make_lead_summary(document, word_limit)
    )

assert result_df[[f"lead_{length}" for length in LEAD_LENGTHS]].notna().all().all()

display(result_df[["title", "summary", "lead_50", "lead_80"]].head(5))

,title,summary,lead_50,lead_80
0,아이트로닉스 차량용 복합기능형 졸음 방지 단말기 특허 출원,"아이일, 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출원했다고 4일 ...",아이엘사이언스의 자회사 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출...,아이엘사이언스의 자회사 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출...
1,뉴스페이스로 가는 한국… 우주기업 성장할 기술·터전 만든다,한국형 우주발사체 누리호가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 성공...,한국형 우주발사체 누리호 가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 날...,한국형 우주발사체 누리호 가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 날...
2,SP500 올 상반기 21%↓…50년래 최악 월가월부,대 대형주 중심의 미국 스탠더드앤드푸어스 S P 500 지수가 올 상반기 50여년만...,닥터둠 루비니 주가 50% 폭락할 것 뉴욕 증시 3대지수 모두 하락 대형주 중심의 ...,닥터둠 루비니 주가 50% 폭락할 것 뉴욕 증시 3대지수 모두 하락 대형주 중심의 ...
3,풀무원 알래스칸 명태살 ‘볼카츠’ 출시,5일 5일 풀무원식품은 알래스카 청정해역의 신선한 명태살로 만든 연육을 동그랗게 빚...,파이낸셜뉴스 풀무원식품은 알래스카 청정해역의 신선한 명태살로 만든 연육을 바삭하게 ...,파이낸셜뉴스 풀무원식품은 알래스카 청정해역의 신선한 명태살로 만든 연육을 바삭하게 ...
4,일동제약 생산본부장에 강덕원 부사장 영입,일동제약은 생산본부장으로 강덕원 부사장을 영입했다고 지난 1일 밝혔으며 강덕원 본부...,강덕원 일동제약 생산본부장. 일동제약 제공 일동제약은 생산본부장으로 강덕원 부사장을...,강덕원 일동제약 생산본부장. 일동제약 제공 일동제약은 생산본부장으로 강덕원 부사장을...


## 4. 한국어 ROUGE 평가

한국어가 제거되는 기본 영문 토큰화 문제를 피하기 위해 모든 베이스라인과 동일하게 공백 기준 토크나이저를 사용합니다.


In [6]:
rouge_metric = evaluate.load("rouge")
references = result_df["summary"].astype(str).tolist()

metric_rows = []
for word_limit in LEAD_LENGTHS:
    prediction_column = f"lead_{word_limit}"
    predictions = result_df[prediction_column].astype(str).tolist()
    scores = rouge_metric.compute(
        predictions=predictions,
        references=references,
        tokenizer=lambda text: text.split(),
        use_stemmer=False,
    )

    prediction_word_counts = result_df[prediction_column].str.split().str.len()
    reference_word_counts = result_df["summary"].str.split().str.len()
    metric_rows.append({
        "strategy": f"lead_{word_limit}",
        "word_limit": word_limit,
        "rouge1": scores["rouge1"] * 100,
        "rouge2": scores["rouge2"] * 100,
        "rougeL": scores["rougeL"] * 100,
        "rougeLsum": scores["rougeLsum"] * 100,
        "average_prediction_words": prediction_word_counts.mean(),
        "average_reference_words": reference_word_counts.mean(),
        "short_document_count": int(prediction_word_counts.lt(word_limit).sum()),
        "evaluated_rows": len(result_df),
    })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(4))

,strategy,word_limit,rouge1,rouge2,rougeL,rougeLsum,average_prediction_words,average_reference_words,short_document_count,evaluated_rows
0,lead_50,50,46.0085,37.6325,39.2564,39.2702,48.0281,42.1496,261,2740
1,lead_80,80,46.4417,39.2197,39.7188,39.7100,74.0088,42.1496,453,2740


## 5. 두 Lead 기준 비교

In [7]:
best_row = metrics_df.loc[metrics_df["rougeL"].idxmax()]
print("ROUGE-L 기준 더 높은 방식:", best_row["strategy"])
print("ROUGE-L:", round(best_row["rougeL"], 4))

comparison_columns = ["title", "summary", "lead_50", "lead_80"]
display(result_df[comparison_columns].head(10))

ROUGE-L 기준 더 높은 방식: lead_80
ROUGE-L: 39.7188


,title,summary,lead_50,lead_80
0,아이트로닉스 차량용 복합기능형 졸음 방지 단말기 특허 출원,"아이일, 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출원했다고 4일 ...",아이엘사이언스의 자회사 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출...,아이엘사이언스의 자회사 아이트로닉스는 차량용 복합기능형 졸음 방지 단말기 특허를 출...
1,뉴스페이스로 가는 한국… 우주기업 성장할 기술·터전 만든다,한국형 우주발사체 누리호가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 성공...,한국형 우주발사체 누리호 가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 날...,한국형 우주발사체 누리호 가 지난 6월 21일 전남 고흥 나로우주센터 발사대에서 날...
2,SP500 올 상반기 21%↓…50년래 최악 월가월부,대 대형주 중심의 미국 스탠더드앤드푸어스 S P 500 지수가 올 상반기 50여년만...,닥터둠 루비니 주가 50% 폭락할 것 뉴욕 증시 3대지수 모두 하락 대형주 중심의 ...,닥터둠 루비니 주가 50% 폭락할 것 뉴욕 증시 3대지수 모두 하락 대형주 중심의 ...
3,풀무원 알래스칸 명태살 ‘볼카츠’ 출시,5일 5일 풀무원식품은 알래스카 청정해역의 신선한 명태살로 만든 연육을 동그랗게 빚...,파이낸셜뉴스 풀무원식품은 알래스카 청정해역의 신선한 명태살로 만든 연육을 바삭하게 ...,파이낸셜뉴스 풀무원식품은 알래스카 청정해역의 신선한 명태살로 만든 연육을 바삭하게 ...
4,일동제약 생산본부장에 강덕원 부사장 영입,일동제약은 생산본부장으로 강덕원 부사장을 영입했다고 지난 1일 밝혔으며 강덕원 본부...,강덕원 일동제약 생산본부장. 일동제약 제공 일동제약은 생산본부장으로 강덕원 부사장을...,강덕원 일동제약 생산본부장. 일동제약 제공 일동제약은 생산본부장으로 강덕원 부사장을...
5,강릉커피산업 발전·애로기술 해결 의기투합…출연기관 업무협약,5일 기술회사인 커피커퍼가 KIST 강릉천연물연구소와 5일 커피 산업 발전을 위한 ...,KIST강릉천연물연구소 등 5일 기업 지원·경제활성화 앞장 약속 한국과학기술연구원 ...,KIST강릉천연물연구소 등 5일 기업 지원·경제활성화 앞장 약속 한국과학기술연구원 ...
6,약세장에 펀드 회전율 5년새 최저…운용기관 방어적 운용 영향,매매에서 주식을 매매하는 빈도를 뜻하는 매매 회전율이 5년 만에 140%까지 떨어진...,매매 수수료 절감 목적도 펀드에서 주식을 매매하는 빈도를 뜻하는 매매 회전율이 5년...,매매 수수료 절감 목적도 펀드에서 주식을 매매하는 빈도를 뜻하는 매매 회전율이 5년...
7,코오롱 수소 밸류체인 플랫폼 구축… 생산부터 발전까지 원스톱,코오롱그룹은 6일 동대문 디자인플라자에서 열린 ‘코리아 H2 비즈니스 서밋 2022...,‘코오롱 H2 플랫폼’ 밸류체인 플랫폼 발표 계열사간 역량 결집과 동시에 대외 파트...,‘코오롱 H2 플랫폼’ 밸류체인 플랫폼 발표 계열사간 역량 결집과 동시에 대외 파트...
8,에너지정책 유턴 재생에너지 대신 원전으로,정부가 정부가 오늘 국무회의에서 의결한 새 에너지 정책 방향의 주요 내용인 원자력 ...,뉴스데스크 앵커 정부가 새 에너지 정책 방향을 오늘 국무회의에서 의결했습니다. 예상...,뉴스데스크 앵커 정부가 새 에너지 정책 방향을 오늘 국무회의에서 의결했습니다. 예상...
9,산업 디지털전환법 오늘부터 시행…“기업 간 협업 프로젝트 지원 확대”,산업디지털전환촉진법 시행을 하루 앞둔 지난 4일 산업부 장영진 제1차관은 산업 디지...,연내 첫 번째 종합계획 수립해 지원사업 본격 추진 이데일리 김형욱 기자 정부가 3년...,연내 첫 번째 종합계획 수립해 지원사업 본격 추진 이데일리 김형욱 기자 정부가 3년...


## 6. 결과 저장

In [8]:
result_df.to_csv(
    RESULTS / "lead_test_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
metrics_df.to_csv(
    RESULTS / "lead_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

run_config = {
    "experiment": "rule-based Lead baseline",
    "dataset_id": DATASET_ID,
    "dataset_revision": dataset_revision,
    "evaluation_split": "original test",
    "test_rows": len(test_df),
    "lead_lengths": LEAD_LENGTHS,
    "unit": "whitespace-separated Korean eojeol",
    "trained": False,
    "summary_used_for_generation": False,
    "summary_used_only_for_evaluation": True,
    "rouge_tokenizer": "whitespace split",
    "seed": SEED,
}
(RESULTS / "run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("저장 완료")
for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PROJECT))

저장 완료
- results\03-3_baseline_lead\lead_metrics.csv
- results\03-3_baseline_lead\lead_test_predictions.csv
- results\03-3_baseline_lead\run_config.json
